# EasyRAG Embeddings Basics

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

What actually becomes the embedding input, and how does similarity search use that representation?

## Success criteria

- inspect chunk text, embedding inputs, and embedding vectors side by side
- compare query-to-chunk similarity scores with deterministic vectors
- connect chunk embeddings to later vector-storage behavior

## Source code anchors

- `easyrag.rag.indexing.pipeline._build_embedding_input`
- `tests.test_rag_retriever._stub_embedding`
- `easyrag.rag.storage.local.EmbeddingVectorStorage._dense_similarity_search`


## Method principles

- `easyrag.rag.indexing.pipeline._build_embedding_input`: This helper builds the object that actually gets embedded. It is where plain text can stay plain text, while multimodal chunks can keep image paths attached for vision-language models.
- `tests.test_rag_retriever._stub_embedding`: This is a deterministic teaching double for embeddings. It turns domain keywords into stable vectors so notebooks and tests can show retrieval behavior without external providers.
- `easyrag.rag.storage.local.EmbeddingVectorStorage._dense_similarity_search`: This internal method is the dense nearest-neighbor path over stored embeddings. It is the local baseline that later notebook comparisons treat as dense retrieval behavior.


## How the code fits together

The flow in this notebook is text input -> embedding vector -> similarity comparison -> backend check. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

import math
from easyrag.rag.indexing.pipeline import build_insert_payloads

## What this cell is proving

We start from a tiny corpus, then inspect the chunk text and the exact embedding inputs that would be handed to the vector layer.

In [ ]:
embed_tmp = tempfile.TemporaryDirectory()
embed_rag = EasyRAG(
    working_dir=embed_tmp.name,
    workspace="embedding-basics-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
documents = EasyRAG.prepare_documents(
    [
        "# Retrieval\nEasyRAG uses query rewrite and grounded retrieval for architecture guidance.\n",
        "# Storage\nRetrieved evidence is packed before answer synthesis.\n",
    ],
    ids=["doc::retrieval", "doc::storage"],
    file_paths=["docs/retrieval.md", "docs/storage.md"],
)
payloads = build_insert_payloads(embed_rag, documents)
query_text = "How does grounded retrieval use query rewrite?"
chunk_inputs = [item["embedding_input"] for item in payloads["vector_chunks"]]
chunk_vectors = _stub_embedding([str(item) for item in chunk_inputs])
query_vector = _stub_embedding([query_text])[0]


def cosine(left, right):
    numerator = sum(a * b for a, b in zip(left, right, strict=False))
    left_norm = math.sqrt(sum(a * a for a in left))
    right_norm = math.sqrt(sum(b * b for b in right))
    return (
        0.0
        if left_norm == 0 or right_norm == 0
        else numerator / (left_norm * right_norm)
    )


rows = []
for item, vector in zip(payloads["vector_chunks"], chunk_vectors, strict=False):
    rows.append(
        {
            "id": item["id"],
            "preview": item["text"][:100].replace("\n", " "),
            "embedding_input": item["embedding_input"],
            "vector": vector,
            "query_similarity": round(cosine(vector, query_vector), 4),
        }
    )
_print_json(rows)

## Why this output looks like this

The next cell stores the vectors in a real workspace so you can compare the manual similarity intuition with an actual `aquery()` result.

In [ ]:
run_sync(embed_rag.initialize_storages())
run_sync(embed_rag.ainsert_documents(documents))
embed_result = run_sync(
    embed_rag.aquery(
        query_text,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
_print_json({"metadata": embed_result.metadata, "citations": embed_result.citations})

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The optional cell asks the configured embedding provider for vectors on the same chunk inputs. The notebook keeps the comparison small: vector length and a preview of the returned values.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-embedding-basics-demo")
    try:
        provider_vectors = provider_rag.embedding_func(chunk_inputs[:2])
        _print_json(
            {
                "vector_lengths": [len(vector) for vector in provider_vectors],
                "first_values": [vector[:5] for vector in provider_vectors],
            }
        )
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

Once a chunk becomes an embedding input, retrieval no longer sees only the raw paragraph. It sees a vectorized representation of that paragraph. That is why inspecting the embedding input itself matters just as much as inspecting the numeric output.

In [ ]:
run_sync(embed_rag.finalize_storages())
embed_tmp.cleanup()
print("Cleaned up the deterministic embedding-basics workspace.")

## Where to go next

- Continue with [03_05_embedding_inputs_and_provider_behavior.ipynb](03_05_embedding_inputs_and_provider_behavior.ipynb) to inspect how different input shapes affect providers.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the indexing-stage map.
- Revisit [03_02_chunking_quality_analysis.ipynb](03_02_chunking_quality_analysis.ipynb) if you want to debug chunk quality before vector quality.